In [1]:
import torch
print("torch:", torch.__version__)
print("cuda :", torch.version.cuda)
print("is_cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))
    print("arch_list:", torch.cuda.get_arch_list())

torch: 2.10.0+cu128
cuda : 12.8
is_cuda_available: True
gpu: Tesla T4
capability: (7, 5)
arch_list: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


In [2]:
# ============================================================
# VIETNAM WILDLIFE CLASSIFIER — TRAINING PIPELINE v6
# ============================================================
# Backbone: BioCLIP ViT-B/16
#   — Pretrained trên 11M ảnh sinh vật học từ iNaturalist
#     (TreeOfLife-10M dataset), đúng domain với dự án này
#   — Tốt hơn EfficientNet pretrain từ ImageNet vì feature
#     space đã được align với ảnh wildlife thực tế
#
# Cải tiến so với v5 (EfficientNet-B4):
#   ✅ BioCLIP ViT-B/16  (hf-hub:imageomics/bioclip)
#   ✅ Progressive unfreezing 3 phase (ViT blocks)
#   ✅ CosineAnnealingWarmRestarts (thoát plateau)
#   ✅ Mixup augmentation  (alpha=0.2, 50% batch)
#   ✅ RandAugment + RandomErasing
#   ✅ Test Time Augmentation (TTA, 4 views)
#   ✅ Sửa lỗi: patience reset đúng sau phase change
#   ✅ Lưu thêm "phase" vào history để plot rõ hơn
# ============================================================

# open_clip_torch cần được cài thêm (không có sẵn trong kaggle)
# Yêu cầu >= 2.26.1 để hỗ trợ cú pháp hf-hub: cho BioCLIP
import subprocess

subprocess.run(
    ["pip", "install", "open_clip_torch>=2.26.1", "-q"], check=True
)

import os
import json
import time
import random

import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, datasets
import open_clip

from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import matplotlib

matplotlib.use("Agg")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# If current PyTorch build does not support this GPU architecture (e.g., P100 sm_60),
# automatically fallback to CPU instead of crashing during first CUDA kernel launch.
ALLOW_CPU_FALLBACK_IF_UNSUPPORTED_GPU = True


# ============================================================
# 0. SEED & DEVICE
# ============================================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    cap_major, cap_minor = torch.cuda.get_device_capability(0)
    if cap_major < 7:
        msg = (
            f"GPU '{gpu_name}' has compute capability sm_{cap_major}{cap_minor}, "
            "but this PyTorch build supports sm_70+ only."
        )
        if ALLOW_CPU_FALLBACK_IF_UNSUPPORTED_GPU:
            print(f"⚠️  {msg}")
            print("⚠️  Falling back to CPU to avoid CUDA runtime crash.")
            print("💡 For GPU training on Kaggle, switch accelerator to T4/L4/A100.")
            DEVICE = torch.device("cpu")
        else:
            raise RuntimeError(
                msg + " Switch Kaggle accelerator to T4/L4/A100, or enable CPU fallback."
            )
    else:
        DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

PIN_MEMORY = DEVICE.type == "cuda"
AMP_ENABLED = DEVICE.type == "cuda"
NUM_WORKERS = min(4, os.cpu_count() or 1)

print(f"✅ Device      : {DEVICE}")
print(
    f"✅ GPU         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}"
)
if torch.cuda.is_available():
    cap_major, cap_minor = torch.cuda.get_device_capability(0)
    print(f"✅ CUDA CC     : sm_{cap_major}{cap_minor}")
print(f"✅ num_workers : {NUM_WORKERS}")
print(f"✅ pin_memory  : {PIN_MEMORY}")

# ============================================================
# 1. CONFIG
# ============================================================
# Input dataset path follows your Kaggle input panel:
# /kaggle/input/wildlife-dataset-preprocessed/wildlife_dataset_processed/splits
PREPROCESSED_ROOT = Path(
    "/kaggle/input/datasets/cduytrn2/wildlife-dataset-preprocessed/wildlife_dataset_processed"
)
SPLIT_DIR = PREPROCESSED_ROOT / "splits"

# Keep outputs in /kaggle/working
MODEL_DIR = Path("/kaggle/working/models_preprocessed")
LOG_DIR = Path("/kaggle/working/logs_preprocessed")

for d in [MODEL_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CFG = {
    # ── Image ──────────────────────────────────────────────
    "img_size": 224,  # CLIP/BioCLIP native size
    # ── Training ───────────────────────────────────────────
    # ViT-B/16 @ 224px nhẹ hơn EfficientNetV2-M @ 480px → batch lớn hơn
    "batch_size": 32,
    "grad_accum": 1,
    "num_epochs": 100,
    "weight_decay": 1e-4,
    "label_smoothing": 0.1,
    "dropout": 0.4,
    "patience": 15,  # dài hơn vì warm restarts
    # ── Progressive Unfreezing (ViT-B/16 có 12 transformer blocks) ──
    # Phase 1 (ep 1..unfreeze_partial_ep-1): chỉ classifier head
    # Phase 2 (ep unfreeze_partial_ep..unfreeze_full_ep-1):
    #         head + resblocks[8..11] (4 block cuối) + ln_post + proj
    # Phase 3 (ep unfreeze_full_ep..): toàn bộ visual encoder
    "unfreeze_partial_ep": 8,
    "unfreeze_full_ep": 18,
    "lr_head": 3e-4,  # phase 1 — chỉ linear head
    "lr_partial": 5e-5,  # phase 2 — 4 blocks cuối
    "lr_full": 1e-5,  # phase 3 — full ViT
    # ── Mixup ──────────────────────────────────────────────
    "mixup_alpha": 0.2,
    "mixup_prob": 0.5,
    # ── TTA ────────────────────────────────────────────────
    "tta_n": 4,  # số augmentation view cho TTA
    # ── Sampling ───────────────────────────────────────────
    "use_weighted_sampler": True,
}

# ============================================================
# 2. HELPERS
# ============================================================
IMG_EXT = {".jpg", ".jpeg", ".png"}


def get_images(directory: Path) -> list:
    return sorted(
        [f for f in directory.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXT]
    )


def rgb_loader(path: str) -> Image.Image:
    with open(path, "rb") as f:
        return Image.open(f).convert("RGB")


# ============================================================
# 3. INPUT VALIDATION
# ============================================================
for split in ["train", "val", "test"]:
    split_path = SPLIT_DIR / split
    if not split_path.exists():
        raise FileNotFoundError(
            f"Missing split folder: {split_path}. "
            "Please check Kaggle input path and dataset version."
        )

print("\n" + "=" * 55)
print("📥 PREPROCESSED INPUT")
print("=" * 55)
print(f"   Root       : {PREPROCESSED_ROOT}")
print(f"   Split root : {SPLIT_DIR}")

# ============================================================
# 5. TRANSFORMS
# ============================================================
# BioCLIP/CLIP dùng normalization riêng, khác ImageNet
_MEAN = [0.48145466, 0.4578275, 0.40821073]
_STD = [0.26862954, 0.26130258, 0.27577711]
_SZ = CFG["img_size"]

train_tf = transforms.Compose(
    [
        transforms.Resize((_SZ + 32, _SZ + 32)),
        transforms.RandomCrop(_SZ),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        transforms.RandomRotation(15, fill=128),
        transforms.RandAugment(num_ops=2, magnitude=9),  # ← mới v6
        transforms.ToTensor(),
        transforms.Normalize(_MEAN, _STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),  # ← mới v6
    ]
)

val_test_tf = transforms.Compose(
    [
        transforms.Resize((_SZ, _SZ)),
        transforms.ToTensor(),
        transforms.Normalize(_MEAN, _STD),
    ]
)

# ── TTA: 4 augmentation views ──────────────────────────────
# Average logits của 4 view → quyết định class cuối
_tta_transforms = [
    # View 1: standard center
    transforms.Compose(
        [
            transforms.Resize((_SZ, _SZ)),
            transforms.ToTensor(),
            transforms.Normalize(_MEAN, _STD),
        ]
    ),
    # View 2: horizontal flip
    transforms.Compose(
        [
            transforms.Resize((_SZ, _SZ)),
            transforms.RandomHorizontalFlip(p=1.0),
            transforms.ToTensor(),
            transforms.Normalize(_MEAN, _STD),
        ]
    ),
    # View 3: scale up → center crop
    transforms.Compose(
        [
            transforms.Resize((int(_SZ * 1.1), int(_SZ * 1.1))),
            transforms.CenterCrop(_SZ),
            transforms.ToTensor(),
            transforms.Normalize(_MEAN, _STD),
        ]
    ),
    # View 4: scale up → center crop + flip
    transforms.Compose(
        [
            transforms.Resize((int(_SZ * 1.1), int(_SZ * 1.1))),
            transforms.CenterCrop(_SZ),
            transforms.RandomHorizontalFlip(p=1.0),
            transforms.ToTensor(),
            transforms.Normalize(_MEAN, _STD),
        ]
    ),
]

# ============================================================
# 6. DATASET & DATALOADERS
# ============================================================
print("\n" + "=" * 55)
print("📦 DATASET")
print("=" * 55)

train_ds = datasets.ImageFolder(
    SPLIT_DIR / "train", transform=train_tf, loader=rgb_loader
)
val_ds = datasets.ImageFolder(
    SPLIT_DIR / "val", transform=val_test_tf, loader=rgb_loader
)
test_ds = datasets.ImageFolder(
    SPLIT_DIR / "test", transform=val_test_tf, loader=rgb_loader
)

assert (
    train_ds.classes == val_ds.classes == test_ds.classes
), "❌ Classes không khớp giữa train/val/test!"

NUM_CLASSES = len(train_ds.classes)
print(f"✅ Số classes : {NUM_CLASSES}")
print(f"   Train      : {len(train_ds):,} ảnh")
print(f"   Val        : {len(val_ds):,} ảnh")
print(f"   Test       : {len(test_ds):,} ảnh")

for split_name in ["train", "val", "test"]:
    split_path = SPLIT_DIR / split_name
    class_sizes = [len(get_images(d)) for d in split_path.iterdir() if d.is_dir()]
    if class_sizes:
        print(
            f"   {split_name:5s} class-size min/median/max: "
            f"{min(class_sizes)}/{int(np.median(class_sizes))}/{max(class_sizes)}"
        )

idx_to_class = {v: k for k, v in train_ds.class_to_idx.items()}
with open(MODEL_DIR / "class_mapping.json", "w") as f:
    json.dump(idx_to_class, f, ensure_ascii=False, indent=2)
print("💾 class_mapping.json saved")

# WeightedRandomSampler — cân bằng class mất cân bằng
targets = [s[1] for s in train_ds.samples]
class_counts = np.bincount(targets)
class_weights = 1.0 / np.where(class_counts == 0, 1, class_counts)
sample_weights = [class_weights[t] for t in targets]
samples_per_class = max(50, len(train_ds) // NUM_CLASSES)
num_samples = NUM_CLASSES * samples_per_class

sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=num_samples, replacement=True
)
print(f"   Sampler    : {num_samples:,} samples/epoch ({samples_per_class}/class)")

if CFG["use_weighted_sampler"]:
    print("   Train loader: WeightedRandomSampler")
    train_loader = DataLoader(
        train_ds,
        batch_size=CFG["batch_size"],
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
else:
    print("   Train loader: shuffle=True (sampler off)")
    train_loader = DataLoader(
        train_ds,
        batch_size=CFG["batch_size"],
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    test_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)


# ============================================================
# 7. MIXUP
# ============================================================
def mixup_data(x, y, alpha: float = 0.2):
    """Trộn 2 sample theo tỉ lệ lam ~ Beta(alpha, alpha)."""
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1.0 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1.0 - lam) * criterion(pred, y_b)


# ============================================================
# 8. MODEL — BioCLIP ViT-B/16
# ============================================================
# BioCLIP (imageomics/bioclip) pretrain trên 11M ảnh sinh vật
# từ TreeOfLife-10M (iNaturalist, GBIF, EOL...) — đúng domain.
#
# open_clip ViT-B/16 visual encoder structure:
#   visual.conv1                     — patch embedding
#   visual.class_embedding           — CLS token
#   visual.positional_embedding      — positional embeddings
#   visual.ln_pre                    — pre-transformer norm
#   visual.transformer.resblocks[0..11] — 12 transformer blocks
#   visual.ln_post                   — post norm  (phase 2 unfreeze)
#   visual.proj                      — 768→512    (phase 2 unfreeze)
#   classifier                       — Dropout + Linear(512, num_classes)


class BioCLIPClassifier(nn.Module):
    """BioCLIP visual encoder + linear classification head."""

    EMBED_DIM = 512  # ViT-B/16 CLIP projected output dim

    def __init__(
        self, visual_encoder: nn.Module, num_classes: int, dropout: float = 0.4
    ):
        super().__init__()
        self.visual = visual_encoder
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(self.EMBED_DIM, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # visual(x) trả về 512-dim projected features, chưa L2-normalize
        features = self.visual(x)
        return self.classifier(features)


print("\n🔄 Đang tải BioCLIP (lần đầu download ~330MB từ HuggingFace)...")
# Với open_clip >= 2.26.1: truyền full hf-hub path vào model_name
# KHÔNG phải pretrained= arg (gây lỗi với version cũ)
_clip_model, _, _ = open_clip.create_model_and_transforms("hf-hub:imageomics/bioclip")

model = BioCLIPClassifier(
    visual_encoder=_clip_model.visual,
    num_classes=NUM_CLASSES,
    dropout=CFG["dropout"],
).to(DEVICE)
del _clip_model  # giải phóng text encoder + CLIP wrapper không cần thiết
torch.cuda.empty_cache()

# Phase 1: freeze toàn bộ visual encoder, chỉ train linear head
for param in model.visual.parameters():
    param.requires_grad = False

scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(
    f"✅ BioCLIP ViT-B/16: {trainable_params:,} / {total_params:,} trainable params (phase 1)"
)

# ============================================================
# 9. LOSS
# ============================================================
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])


# ============================================================
# 10. OPTIMIZER & SCHEDULER HELPERS
# ============================================================
def make_optimizer(model: nn.Module, lr: float) -> optim.Optimizer:
    return optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=CFG["weight_decay"],
    )


def make_scheduler(
    optimizer: optim.Optimizer, T_0: int
) -> optim.lr_scheduler.LRScheduler:
    """CosineAnnealingWarmRestarts: restart mỗi T_0 epoch, chu kỳ nhân đôi."""
    return optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=max(T_0, 5),
        T_mult=2,
        eta_min=1e-7,
    )


optimizer = make_optimizer(model, CFG["lr_head"])
scheduler = make_scheduler(optimizer, T_0=CFG["unfreeze_partial_ep"] - 1)

# ============================================================
# 11. PROGRESSIVE UNFREEZE
# ============================================================
CURRENT_PHASE = 1


def set_phase(new_phase: int, current_epoch: int):
    """
    Chuyển phase: thay đổi requires_grad của ViT blocks, rebuild optimizer + scheduler.

    Phase 1: chỉ classifier head  (frozen: toàn bộ visual)
    Phase 2: head + resblocks[8..11] + ln_post + proj  (4 blocks cuối)
    Phase 3: toàn bộ model  (full fine-tune)
    """
    global optimizer, scheduler, CURRENT_PHASE

    if new_phase == CURRENT_PHASE:
        return

    print(f"\n🔓 Phase {CURRENT_PHASE} → Phase {new_phase}  (epoch {current_epoch})")

    if new_phase == 2:
        # Freeze lại tất cả visual trước
        for p in model.visual.parameters():
            p.requires_grad = False
        # Unfreeze 4 blocks cuối (resblocks[8..11]), ln_post, proj
        for blk in model.visual.transformer.resblocks[8:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in model.visual.ln_post.parameters():
            p.requires_grad = True
        if model.visual.proj is not None:
            model.visual.proj.requires_grad = True
        lr = CFG["lr_partial"]
        T_0 = CFG["unfreeze_full_ep"] - CFG["unfreeze_partial_ep"]

    elif new_phase == 3:
        # Unfreeze toàn bộ visual encoder + classifier
        for p in model.parameters():
            p.requires_grad = True
        lr = CFG["lr_full"]
        T_0 = CFG["num_epochs"] - CFG["unfreeze_full_ep"]

    optimizer = make_optimizer(model, lr)
    scheduler = make_scheduler(optimizer, T_0=T_0)

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Trainable params : {n_trainable:,}")
    print(f"   LR               : {lr:.1e}")
    print(f"   Scheduler T_0    : {max(T_0, 5)}\n")

    CURRENT_PHASE = new_phase


# ============================================================
# 12. CHECKPOINT
# ============================================================
CKPT_PATH = MODEL_DIR / "checkpoint.pth"
BEST_PATH = MODEL_DIR / "best_model.pth"


def save_checkpoint(
    epoch, val_acc, best_val_acc, no_improve, phase, history, is_best=False
):
    state = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "val_acc": val_acc,
        "best_val_acc": best_val_acc,
        "no_improve": no_improve,
        "phase": phase,
        "history": history,
        "num_classes": NUM_CLASSES,
        "cfg": CFG,
        "idx_to_class": idx_to_class,
    }
    torch.save(state, CKPT_PATH)
    if is_best:
        torch.save(state, BEST_PATH)


def load_checkpoint():
    global optimizer, scheduler, CURRENT_PHASE
    if not CKPT_PATH.exists():
        return None

    print("\n🔄 Phát hiện checkpoint — resume...")
    torch.cuda.empty_cache()
    ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

    model.load_state_dict(ckpt["model_state"])
    model.to(DEVICE)

    saved_phase = ckpt.get("phase", 1)
    if saved_phase == 2:
        for p in model.visual.parameters():
            p.requires_grad = False
        for blk in model.visual.transformer.resblocks[8:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in model.visual.ln_post.parameters():
            p.requires_grad = True
        if model.visual.proj is not None:
            model.visual.proj.requires_grad = True
        lr = CFG["lr_partial"]
        T_0 = CFG["unfreeze_full_ep"] - CFG["unfreeze_partial_ep"]
    elif saved_phase == 3:
        for p in model.parameters():
            p.requires_grad = True
        lr = CFG["lr_full"]
        T_0 = CFG["num_epochs"] - CFG["unfreeze_full_ep"]
    else:
        # phase 1: chỉ classifier, freeze toàn bộ visual
        for p in model.visual.parameters():
            p.requires_grad = False
        lr = CFG["lr_head"]
        T_0 = CFG["unfreeze_partial_ep"] - 1

    optimizer = make_optimizer(model, lr)
    scheduler = make_scheduler(optimizer, T_0=T_0)
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    CURRENT_PHASE = saved_phase

    print(
        f"   ✅ Resume từ epoch {ckpt['epoch']} "
        f"| phase={saved_phase} | best_val={ckpt['best_val_acc']:.2%}"
    )
    return ckpt


# ============================================================
# 13. TRAIN / EVAL FUNCTIONS
# ============================================================
from tqdm import tqdm


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    run_loss, correct, n_total = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(
        loader, desc="  Train", leave=False, bar_format="{l_bar}{bar:20}{r_bar}"
    )

    for step, (imgs, labels) in enumerate(pbar):
        imgs = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        # Mixup với xác suất mixup_prob
        use_mixup = random.random() < CFG["mixup_prob"]
        if use_mixup:
            imgs, y_a, y_b, lam = mixup_data(imgs, labels, CFG["mixup_alpha"])

        with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
            outputs = model(imgs)
            if use_mixup:
                loss = mixup_criterion(criterion, outputs, y_a, y_b, lam)
            else:
                loss = criterion(outputs, labels)
            loss = loss / CFG["grad_accum"]

        scaler.scale(loss).backward()
        run_loss += loss.item() * CFG["grad_accum"] * imgs.size(0)

        # Accuracy dùng ground-truth (xấp xỉ khi có mixup — chấp nhận được cho logging)
        with torch.no_grad():
            correct += (outputs.argmax(1) == labels).sum().item()
        n_total += imgs.size(0)

        if (step + 1) % CFG["grad_accum"] == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        pbar.set_postfix(loss=f"{run_loss/n_total:.4f}", acc=f"{correct/n_total:.2%}")

    return run_loss / n_total, correct / n_total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    run_loss, correct, n_total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        run_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        n_total += imgs.size(0)

    return run_loss / n_total, correct / n_total


# ============================================================
# 14. TTA EVALUATION
# ============================================================
@torch.no_grad()
def evaluate_tta(model, test_dir: Path, tta_transforms: list, batch_size: int):
    """
    Chạy inference N lần với các augmentation khác nhau trên cùng test set.
    Average logits rồi tính top-1 và top-3.
    """
    model.eval()

    # Lấy ground truth từ lần đầu
    ref_ds = datasets.ImageFolder(
        test_dir, transform=tta_transforms[0], loader=rgb_loader
    )
    n_samples = len(ref_ds)
    true_labels = torch.tensor([s[1] for s in ref_ds.samples])
    all_logits = torch.zeros(n_samples, NUM_CLASSES)  # tích lũy logits

    for idx, tf in enumerate(tta_transforms):
        print(f"   TTA view {idx + 1}/{len(tta_transforms)} ...", end=" ", flush=True)
        tmp_ds = datasets.ImageFolder(test_dir, transform=tf, loader=rgb_loader)
        tmp_ld = DataLoader(
            tmp_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        )
        offset = 0
        for imgs, _ in tmp_ld:
            imgs = imgs.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
                logits = model(imgs)
            # .float() để tránh half-precision accumulation error
            all_logits[offset : offset + imgs.size(0)] += logits.cpu().float()
            offset += imgs.size(0)
        print("✓")

    preds_top1 = all_logits.argmax(dim=1)
    preds_top3 = all_logits.topk(3, dim=1).indices

    top1_acc = (preds_top1 == true_labels).float().mean().item()
    top3_acc = (preds_top3 == true_labels.unsqueeze(1)).any(dim=1).float().mean().item()

    return top1_acc, top3_acc, preds_top1.numpy(), true_labels.numpy()


# ============================================================
# 15. TRAINING LOOP
# ============================================================
ckpt = load_checkpoint()

if ckpt:
    start_epoch = ckpt["epoch"] + 1
    best_val_acc = ckpt["best_val_acc"]
    best_epoch = ckpt["epoch"]
    no_improve = ckpt["no_improve"]
    history = ckpt["history"]
    CURRENT_PHASE = ckpt.get("phase", 1)
else:
    start_epoch = 1
    best_val_acc = 0.0
    best_epoch = 0
    no_improve = 0
    history = {
        k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "lr", "phase"]
    }

print("\n" + "=" * 65)
print("🚀 TRAINING")
print("=" * 65)
print(
    f"{'Ep':>3} | {'TrLoss':>7} | {'TrAcc':>6} | "
    f"{'VaLoss':>7} | {'VaAcc':>6} | {'LR':>8} | {'Ph'} | {'s':>4}"
)
print("─" * 65)

for epoch in range(start_epoch, CFG["num_epochs"] + 1):

    # ── Progressive unfreeze ─────────────────────────────────
    if epoch >= CFG["unfreeze_full_ep"] and CURRENT_PHASE < 3:
        set_phase(3, epoch)
        no_improve = 0  # reset patience khi thay đổi phase

    elif epoch >= CFG["unfreeze_partial_ep"] and CURRENT_PHASE < 2:
        set_phase(2, epoch)
        no_improve = 0

    # ── Train + Eval ─────────────────────────────────────────
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    scheduler.step()

    lr = optimizer.param_groups[0]["lr"]
    elapsed = time.time() - t0

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(lr)
    history["phase"].append(CURRENT_PHASE)

    # ── Checkpoint ───────────────────────────────────────────
    improved = val_acc > best_val_acc
    flag = " 💾" if improved else ""

    print(
        f"{epoch:>3} | {train_loss:>7.4f} | {train_acc:>5.2%} | "
        f"{val_loss:>7.4f} | {val_acc:>5.2%} | "
        f"{lr:>8.2e} | {CURRENT_PHASE:>2} | {elapsed:>3.0f}s{flag}"
    )

    if improved:
        best_val_acc = val_acc
        best_epoch = epoch
        no_improve = 0
    else:
        no_improve += 1

    save_checkpoint(
        epoch,
        val_acc,
        best_val_acc,
        no_improve,
        CURRENT_PHASE,
        history,
        is_best=improved,
    )

    # ── Early Stopping ───────────────────────────────────────
    if no_improve >= CFG["patience"]:
        print(
            f"\n⏹️  Early stopping tại epoch {epoch} "
            f"(+{CFG['patience']} epochs không cải thiện)"
        )
        break

print(f"\n🏆 Best: epoch {best_epoch} — val_acc = {best_val_acc:.2%}")

# ============================================================
# 16. ĐÁNH GIÁ CUỐI — STANDARD + TTA
# ============================================================
print("\n" + "=" * 55)
print("📊 EVALUATION")
print("=" * 55)

# Load best model
ckpt_best = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt_best["model_state"])

# Standard eval (không TTA)
_, std_top1 = evaluate(model, test_loader, criterion)
print(f"✅ Test Top-1 (standard) : {std_top1:.2%}")

# TTA eval
print(f"\n🔄 TTA evaluation ({CFG['tta_n']} views)...")
tta_top1, tta_top3, all_preds, all_labels = evaluate_tta(
    model,
    SPLIT_DIR / "test",
    _tta_transforms[: CFG["tta_n"]],
    batch_size=CFG["batch_size"],
)
print(
    f"✅ Test Top-1 (TTA)      : {tta_top1:.2%}  "
    f"(+{(tta_top1 - std_top1) * 100:.2f}% vs standard)"
)
print(f"✅ Test Top-3 (TTA)      : {tta_top3:.2%}")

# Classification report dùng kết quả TTA
class_names = [idx_to_class[i] for i in range(NUM_CLASSES)]
report_dict = classification_report(
    all_labels,
    all_preds,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(report_dict).T.to_csv(LOG_DIR / "classification_report.csv")
print("💾 classification_report.csv saved")

# ============================================================
# 17. PLOTS
# ============================================================
ep_range = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, tk, vk, title in [
    (axes[0], "train_loss", "val_loss", "Loss"),
    (axes[1], "train_acc", "val_acc", "Accuracy"),
]:
    ax.plot(ep_range, history[tk], label="Train")
    ax.plot(ep_range, history[vk], label="Val")
    # Vẽ đường đánh dấu thay đổi phase
    phases = history.get("phase", [])
    for ep_idx, ph in enumerate(phases[1:], start=2):
        if phases[ep_idx - 2] != ph:
            ax.axvline(
                ep_idx, color="orange", linestyle=":", alpha=0.8, label=f"→ Phase {ph}"
            )
    if best_epoch:
        ax.axvline(
            best_epoch,
            color="green",
            linestyle="--",
            alpha=0.8,
            label=f"Best (ep{best_epoch})",
        )
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[2].plot(ep_range, history["lr"])
axes[2].set_title("Learning Rate")
axes[2].set_xlabel("Epoch")
axes[2].set_yscale("log")
axes[2].grid(True, alpha=0.3)

plt.suptitle(
    f"Vietnam Wildlife v6 — BioCLIP ViT-B/16  |  "
    f"Best Val: {best_val_acc:.2%} (ep{best_epoch})  |  "
    f"TTA Top-1: {tta_top1:.2%}  |  TTA Top-3: {tta_top3:.2%}",
    fontsize=11,
)
plt.tight_layout()
plt.savefig(LOG_DIR / "training_curves.png", dpi=150)
print("💾 training_curves.png saved")

# ============================================================
# 18. SAVE HISTORY & SUMMARY
# ============================================================
pd.DataFrame(history).to_csv(LOG_DIR / "training_history.csv", index=False)

summary = {
    "model": "BioCLIP-ViT-B16",
    "best_epoch": best_epoch,
    "best_val_acc": round(best_val_acc, 4),
    "test_top1_standard": round(std_top1, 4),
    "test_top1_tta": round(tta_top1, 4),
    "test_top3_tta": round(tta_top3, 4),
    "num_classes": NUM_CLASSES,
    "total_params": total_params,
    "img_size": CFG["img_size"],
    "epochs_ran": len(history["train_loss"]),
    "early_stopped": no_improve >= CFG["patience"],
    "tta_n": CFG["tta_n"],
}
with open(LOG_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 55)
print("✅ HOÀN THÀNH!")
print("=" * 55)
print(f"   Best Val Acc  : {best_val_acc:.2%} (epoch {best_epoch})")
print(f"   Top-1 standard: {std_top1:.2%}")
print(f"   Top-1 TTA     : {tta_top1:.2%}")
print(f"   Top-3 TTA     : {tta_top3:.2%}")
print(f"   Epochs ran    : {len(history['train_loss'])}/{CFG['num_epochs']}")
print(f"\n📁 Output:")
print(f"   {MODEL_DIR}/best_model.pth")
print(f"   {MODEL_DIR}/checkpoint.pth")
print(f"   {MODEL_DIR}/class_mapping.json")
print(f"   {LOG_DIR}/training_curves.png")
print(f"   {LOG_DIR}/classification_report.csv")
print(f"   {LOG_DIR}/training_history.csv")
print(f"   {LOG_DIR}/summary.json")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
✅ Device      : cuda
✅ GPU         : Tesla T4
✅ CUDA CC     : sm_75
✅ num_workers : 4
✅ pin_memory  : True

📥 PREPROCESSED INPUT
   Root       : /kaggle/input/datasets/cduytrn2/wildlife-dataset-preprocessed/wildlife_dataset_processed
   Split root : /kaggle/input/datasets/cduytrn2/wildlife-dataset-preprocessed/wildlife_dataset_processed/splits

📦 DATASET
✅ Số classes : 570
   Train      : 91,200 ảnh
   Val        : 10,887 ảnh
   Test       : 10,887 ảnh
   train class-size min/median/max: 160/160/160
   val   class-size min/median/max: 12/20/20
   test  class-size min/median/max: 12/20/20
💾 class_mapping.json saved
   Sampler    : 91,200 samples/epoch (160/class)
   Train loader: WeightedRandomSampler

🔄 Đang tải BioCLIP (lần đầu download ~330MB từ HuggingFace)...


open_clip_config.json:   0%|          | 0.00/469 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

✅ BioCLIP ViT-B/16: 292,410 / 86,485,050 trainable params (phase 1)

🚀 TRAINING
 Ep |  TrLoss |  TrAcc |  VaLoss |  VaAcc |       LR | Ph |    s
─────────────────────────────────────────────────────────────────


  1 |  5.1317 | 18.58% |  3.3150 | 55.78% | 2.85e-04 |  1 | 574s 💾


...

 28 |  1.6390 | 71.42% |  1.6215 | 83.50% | 9.57e-06 |  3 | 926s


 29 |  1.6082 | 71.20% |  1.6215 | 83.51% | 9.49e-06 |  3 | 925s


 30 |  1.6410 | 70.68% |  1.6179 | 83.72% | 9.40e-06 |  3 | 926s


 31 |  1.6260 | 69.56% |  1.6247 | 83.63% | 9.30e-06 |  3 | 924s


 32 |  1.5752 | 69.85% |  1.6250 | 83.91% | 9.20e-06 |  3 | 926s

⏹️  Early stopping tại epoch 32 (+15 epochs không cải thiện)

🏆 Best: epoch 17 — val_acc = 84.23%

📊 EVALUATION
✅ Test Top-1 (standard) : 84.46%

🔄 TTA evaluation (4 views)...
   TTA view 1/4 ... ✓
   TTA view 2/4 ... ✓
   TTA view 3/4 ... ✓
   TTA view 4/4 ... ✓
✅ Test Top-1 (TTA)      : 85.95%  (+1.49% vs standard)
✅ Test Top-3 (TTA)      : 94.09%
💾 classification_report.csv saved
💾 training_curves.png saved

✅ HOÀN THÀNH!
   Best Val Acc  : 84.23% (epoch 17)
   Top-1 standard: 84.46%
   Top-1 TTA     : 85.95%
   Top-3 TTA     : 94.09%
   Epochs ran    : 32/100

📁 Output:
   /kaggle/working/models_preprocessed/best_model.pth
   /kaggle/working/models_preprocessed/checkpoint.pth
   /kaggle/working/models_preprocessed/class_mapping.json
   /kaggle/working/logs_preprocessed/training_curves.png
   /kaggle/working/logs_preprocessed/classification_report.csv
   /kaggle/working/logs_preprocessed/training_history.csv
   /kaggl